# Notebook 01 · 骨架：双层相依网络与级联循环

**这一节要建立的三件事**

1. 双层网络怎么在代码里表示：`G['A']`、`G['B']` 两张图 + `partner` 耦合字典；
2. 级联为什么是"求不动点"：判失效 → 剪 GCC → 再判，直到没有人新失效；
3. 层间依赖为什么致命：伙伴死了我也死（布尔耦合）。

对齐文献：**Buldyrev et al., Nature 464:1025 (2010)** 是最早提出"相依网络级联"的论文，
**段东立 et al., PNAS 116(45):22452 (2019)** 是本项目的理论总锚（自动力学 + 容忍判据）。

## 环境准备（每个 notebook 都要先跑这两格）

In [ ]:
"""环境准备：定位项目根目录、加 src 到搜索路径、设好画图参数。"""
import sys
from pathlib import Path

# 从当前工作目录往上找，直到看见 src/phase6_interdependent（这样在 notebooks/ 里也能跑）
root = Path.cwd()
while not (root / "src" / "phase6_interdependent").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

RESULTS = root / "results"
RESULTS.mkdir(exist_ok=True)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("项目根目录：", root)

In [ ]:
# 公共底座：一对 BA 双层网络 + 骨架类（后面所有 notebook 都从这一格开始）
from phase6_interdependent.core import InterdependentNetwork

N, m = 300, 3
G_A = nx.barabasi_albert_graph(N, m, seed=42)   # 层 A：比如电力网
G_B = nx.barabasi_albert_graph(N, m, seed=43)   # 层 B：比如通信网
nodes_A = sorted(G_A.nodes())
print(f"双层网络就绪：N={N}, 每层边数={G_A.number_of_edges()}")

## 第 1 步

In [ ]:
from phase6_interdependent.core import InterdependentNetwork

N, m = 300, 3
G_A = nx.barabasi_albert_graph(N, m, seed=42)   # 层 A：比如电力网
G_B = nx.barabasi_albert_graph(N, m, seed=43)   # 层 B：比如通信网
net = InterdependentNetwork(G_A, G_B)
print("层 A 节点数：", net.n_nodes("A"), " 层 B 节点数：", net.n_nodes("B"))
print("耦合对数：", len(net.partner["A"]), "（一一对应）")
print("初始存活序参量 S_A =", net.S("A"))

## 第 2 步

In [ ]:
# 随机移除 5% 的 A 层节点，看级联怎么扩散
rng = np.random.default_rng(0)
failed = rng.choice(sorted(G_A.nodes()), size=int(0.05 * N), replace=False)
res = net.run(failed, record=True)
print(f"初始故障 {res['failed_initial']} 个 → 终态 S_A={res['S_A']:.3f}, "
      f"S_B={res['S_B']:.3f}, 级联步数={res['steps']}")

hist = [h["S_A"] for h in res["history"]]
plt.figure(figsize=(7, 4))
plt.plot(range(len(hist)), hist, "o-")
plt.xlabel("级联轮次")
plt.ylabel("层 A 存活序参量 S")
plt.title("一次级联的演化过程（不动点迭代）")
plt.grid(alpha=0.3)
plt.savefig(RESULTS / "notebook_01_cascade_trace.png", dpi=150, bbox_inches="tight")
plt.show()

## 第 3 步

In [ ]:
# 对照实验：移除 hub（度最大的节点） vs 移除叶子节点
hub = max(G_A.degree(), key=lambda kv: kv[1])[0]
leaf = min(G_A.degree(), key=lambda kv: kv[1])[0]

for label, victim in [("hub", hub), ("leaf", leaf)]:
    n2 = InterdependentNetwork(G_A, G_B)
    r = n2.run([victim])
    print(f"移除 {label:>4} 节点 {victim:>4} → S_A={r['S_A']:.3f}, S_B={r['S_B']:.3f}")

**思考题**

1. 为什么"一对一的层间耦合"会让局部故障变成全网故障？
2. 把 `coupled_fraction` 从 1.0 降到 0.2 会发生什么？（下一节 L0 的正式实验）
3. `net.run` 里的 `max_steps` 是干什么的？没有它会出什么问题？